In [0]:

# ============================================================
# CELDA 1. VALIDACIÓN DE CALIDAD DE BRONZE
# ============================================================

# Revisamos las dos fuentes del proyecto:
# 1. BTS: registros originales de vuelos de 2022 a 2025.
# 2. Open-Meteo: pronósticos horarios de 30 aeropuertos.
#
# Comprobamos cobertura, metadatos y datos faltantes.
# No limpiamos ni modificamos las tablas originales.

from pyspark.sql import functions as F
import calendar
import pandas as pd

TABLA_BTS = "workspace.bronze_flights.bts_flights_raw"
TABLA_CLIMA = "workspace.bronze_flights.weather_forecasts_raw"
TABLA_TOP30 = "workspace.bronze_flights.weather_airports_top30"

# ============================================================
# 1. VALIDACIÓN DE BTS
# ============================================================

print("=" * 55)
print("1. VALIDACIÓN DE BRONZE BTS")
print("=" * 55)

bts = spark.table(TABLA_BTS)

# Comprobamos los registros por año y mes.
resumen_bts = (
    bts.groupBy("_source_year", "_source_month")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("_source_file").alias("archivos"),
        F.countDistinct("_ingestion_timestamp")
        .alias("marcas_ingesta")
    )
)

periodos_bts = {
    (fila["_source_year"], fila["_source_month"])
    for fila in resumen_bts.collect()
}

esperados_bts = {
    (anio, mes)
    for anio in range(2022, 2026)
    for mes in range(1, 13)
}

print("Periodos encontrados:", len(periodos_bts), "de 48")
print("Periodos faltantes:", len(esperados_bts - periodos_bts))
print("Total de registros:", f"{bts.count():,}")

# Verificamos que los metadatos estén completos.
nulos_metadata_bts = bts.filter(
    F.col("_source_year").isNull() |
    F.col("_source_month").isNull() |
    F.col("_source_file").isNull() |
    F.col("_ingestion_timestamp").isNull()
).count()

print("Registros sin metadatos:", nulos_metadata_bts)

# Revisamos la disponibilidad de variables importantes.
# En Bronze es normal conservar nulos originales.
columnas_bts = [
    "FLIGHTDATE",
    "ORIGIN",
    "DEST",
    "CRSDEPTIME",
    "DEPDELAY",
    "CANCELLED"
]

nulos_bts = bts.agg(*[
    F.sum(
        F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1)
        .otherwise(0)
    ).alias(c)
    for c in columnas_bts
]).first().asDict()

print("\nNULOS BTS EN VARIABLES PRINCIPALES")
display(pd.DataFrame(
    list(nulos_bts.items()),
    columns=["Variable", "Registros nulos"]
))

# ============================================================
# 2. VALIDACIÓN DE OPEN-METEO
# ============================================================

print("\n" + "=" * 55)
print("2. VALIDACIÓN DE BRONZE METEOROLÓGICO")
print("=" * 55)

clima = spark.table(TABLA_CLIMA)

codigos = [
    fila["airport"]
    for fila in spark.table(TABLA_TOP30)
    .select("airport").collect()
]

# Analizamos únicamente los aeropuertos y el periodo
# definidos para nuestro modelo meteorológico.
clima_modelo = (
    clima.filter(F.col("airport").isin(codigos))
    .filter(F.col("source_model") == "gfs_global")
    .filter(F.col("forecast_lead_hours") == 24)
    .filter(
        (F.col("source_year") == 2025) |
        (
            (F.col("source_year") == 2024) &
            (F.col("source_month") >= 2)
        )
    )
)

print("Aeropuertos:", len(codigos))
print("Registros meteorológicos:", f"{clima_modelo.count():,}")

# Comprobamos la cantidad de periodos aeropuerto-mes.
resumen_clima = (
    clima_modelo
    .groupBy("airport", "source_year", "source_month")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("forecast_valid_time_utc")
        .alias("horas_unicas")
    )
)

print("Periodos aeropuerto-mes:", resumen_clima.count(), "de 690")

# Comprobamos si existen registros duplicados
# por aeropuerto y hora dentro de cada periodo.
duplicados = resumen_clima.filter(
    F.col("registros") != F.col("horas_unicas")
).count()

print("Periodos con duplicados:", duplicados)

# Revisamos nulos en las seis variables meteorológicas.
variables_clima = [
    "temperature_2m_previous_day1",
    "precipitation_previous_day1",
    "wind_speed_10m_previous_day1",
    "relative_humidity_2m_previous_day1",
    "cloud_cover_previous_day1",
    "pressure_msl_previous_day1"
]

nulos_clima = clima_modelo.agg(*[
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in variables_clima
]).first().asDict()

print("\nNULOS EN VARIABLES METEOROLÓGICAS")
display(pd.DataFrame(
    list(nulos_clima.items()),
    columns=["Variable", "Registros nulos"]
))

# ============================================================
# 3. RESULTADO GENERAL
# ============================================================

print("\n" + "=" * 55)
print("RESUMEN DE VALIDACIÓN BRONZE")
print("=" * 55)

print("BTS: cobertura de", len(periodos_bts), "de 48 meses.")
print("BTS: metadatos faltantes:", nulos_metadata_bts)
print("Clima: cobertura de", resumen_clima.count(), "de 690 periodos.")
print("Clima: periodos con duplicados:", duplicados)

# Los nulos originales de BTS se reportan, pero no
# se consideran automáticamente un error de ingesta.
if (
    periodos_bts == esperados_bts
    and nulos_metadata_bts == 0
    and resumen_clima.count() == 690
    and duplicados == 0
    and all(v == 0 for v in nulos_clima.values())
):
    print("VALIDACIÓN GENERAL EXITOSA.")
else:
    print("REVISAR: existen inconsistencias o datos faltantes.")


In [0]:

# ============================================================
# CELDA 2. REVISAR NULOS DE NUESTRA VARIABLE OBJETIVO
# ============================================================

# DEPDELAY representa los minutos de retraso de salida.
# Queremos predecir si el vuelo tendrá 15 minutos o más
# de retraso, por lo que debemos investigar sus nulos.
#
# Revisamos cuántos corresponden a vuelos cancelados
# y cuántos a vuelos que no fueron cancelados.

from pyspark.sql import functions as F

TABLA_BTS = "workspace.bronze_flights.bts_flights_raw"

df = spark.table(TABLA_BTS)

# Convertimos los campos a valores numéricos.
df = (
    df
    .withColumn(
        "dep_delay_num",
        F.col("DEPDELAY").cast("double")
    )
    .withColumn(
        "cancelled_num",
        F.col("CANCELLED").cast("double")
    )
)

# Clasificamos los registros según su situación.
revision = (
    df
    .withColumn(
        "situacion",
        F.when(
            (F.col("cancelled_num") == 1) &
            F.col("dep_delay_num").isNull(),
            "Cancelado sin DEPDELAY"
        )
        .when(
            (F.col("cancelled_num") == 0) &
            F.col("dep_delay_num").isNull(),
            "No cancelado sin DEPDELAY"
        )
        .when(
            F.col("dep_delay_num").isNotNull(),
            "Con DEPDELAY disponible"
        )
        .otherwise("Otro caso")
    )
    .groupBy("situacion")
    .agg(F.count("*").alias("total_vuelos"))
    .orderBy(F.desc("total_vuelos"))
)

# Calculamos porcentajes sobre el total de vuelos.
total = df.count()

revision = revision.withColumn(
    "porcentaje",
    F.round(
        F.col("total_vuelos") / F.lit(total) * 100,
        2
    )
)

print("REVISIÓN DE LA VARIABLE OBJETIVO")
print("Total de vuelos:", f"{total:,}")

display(revision)


### Conclusiones de calidad de datos Bronze

Se validaron las dos fuentes principales del proyecto: BTS y Open-Meteo.

La tabla BTS contiene información de los 48 meses de 2022 a 2025, sin periodos faltantes ni problemas en los metadatos de ingesta. En la revisión de la variable `DEPDELAY` encontramos 453,438 valores nulos, equivalentes aproximadamente al 1.64% de los registros. Todos los valores nulos identificados corresponden a vuelos cancelados.

Por su parte, la tabla meteorológica contiene 504,000 registros de pronósticos horarios para 30 aeropuertos, distribuidos en 690 combinaciones de aeropuerto y mes. No se encontraron duplicados ni valores faltantes en las seis variables meteorológicas revisadas.

Para el modelo de predicción de retrasos de al menos 15 minutos, los vuelos cancelados se excluirán del conjunto de entrenamiento, ya que no tienen un retraso de salida comparable. La limpieza definitiva se realizará en Silver, sin modificar los registros originales almacenados en Bronze.


In [0]:
print("Total real BTS:", spark.table(
    "workspace.bronze_flights.bts_flights_raw"
).count())